# Reproduce CEMC and Li Percolation Analysis for Li1.2Mn0.4Ti0.4O2

This notebook provides an end-to-end example for reproducing the CEMC and percolation workflow used for the Li1.2Mn0.4Ti0.4O2 composition. It generates 100 CEMC structures at each of 100 K, 1000 K, 10000 K, and 100000 K, then calculates the percolating Li fraction for each structure. The final panel follows the Fig. 2b analysis workflow: percolating Li fraction versus CEMC temperature.

The full run performs 400 CEMC simulations and can take substantial time on a workstation. To test the workflow quickly, reduce `N_STRUCTURES` in the configuration cell.

## Dependencies

Install the study-specific and external packages before running the notebook:

```bash
pip install git+https://github.com/Liaojh123/SROS.git
pip install git+https://github.com/CederGroupHub/smol.git
pip install git+https://github.com/atomisticnet/dribble.git
pip install pymatgen numpy pandas matplotlib
```

`SROS` is used to build the initial DRX structures and call the CEMC workflow, `smol` is required by the cluster-expansion Monte Carlo code, and `dribble` is used for the Li percolation analysis.

In [ ]:
from __future__ import annotations

import inspect
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from SROS import CEMC, generate_random_any_composition
from dribble.io import Input
from dribble.lattice import Lattice
from dribble.percolator import Percolator


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    marker = Path("models") / "Li-Mn-Ti-O_clustre_expansion_model.mson"
    for path in (start, *start.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError("Could not find repository root containing models/Li-Mn-Ti-O_clustre_expansion_model.mson")


REPO_ROOT = find_repo_root()
CE_MODEL_PATH = REPO_ROOT / "models" / "Li-Mn-Ti-O_clustre_expansion_model.mson"
OUTPUT_DIR = REPO_ROOT / "outputs" / "cemc_percolation_Li1p2Mn0p4Ti0p4O2"

LI_CONTENT = 1.2
MN_CONTENT = 0.4
TI_CONTENT = 2.0 - LI_CONTENT - MN_CONTENT
COMPOSITION = "Li1.2Mn0.4Ti0.4O2"

TEMPERATURES = [100, 1000, 10000, 100000]
N_STRUCTURES = 100
MC_STEPS = 120000
SC_MATRIX = np.array([[5, 0, 0], [0, 6, 0], [0, 0, 8]])
BASE_SEED = 202502

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"CE model: {CE_MODEL_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Composition: {COMPOSITION}; Ti content = {TI_CONTENT:.1f}")


## CEMC structure generation

For each target temperature, the notebook creates random Li-Mn-Ti-O DRX starting structures using `SROS.generate_random_any_composition`, runs CEMC using the provided Li-Mn-Ti-O cluster-expansion model, and writes the final CEMC structure to `outputs/`.

In [ ]:
def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def make_initial_structure(seed: int):
    set_random_seed(seed)
    supercell = generate_random_any_composition.make_supercell(scaling_matrix=SC_MATRIX.tolist())
    return generate_random_any_composition.modify_structure(LI_CONTENT, MN_CONTENT, supercell)


def run_cemc(initial_structure, temperature: int):
    kwargs = {"sc_matrix": SC_MATRIX, "mc_step": MC_STEPS}
    signature = inspect.signature(CEMC.run_cemc)
    if "temperature" in signature.parameters:
        kwargs["temperature"] = temperature
    elif "T" in signature.parameters:
        kwargs["T"] = temperature
    elif "mc_temperature" in signature.parameters:
        kwargs["mc_temperature"] = temperature
    else:
        print("Warning: CEMC.run_cemc does not expose a recognized temperature keyword; using its package default.")

    return CEMC.run_cemc(
        LI_CONTENT,
        MN_CONTENT,
        initial_structure,
        str(CE_MODEL_PATH),
        **kwargs,
    )


def structure_path(temperature: int, index: int) -> Path:
    return OUTPUT_DIR / f"{temperature}K" / "structures" / f"CEMC_{index:03d}.vasp"


def generate_cemc_structures(overwrite: bool = False) -> pd.DataFrame:
    records = []
    for temperature in TEMPERATURES:
        for index in range(N_STRUCTURES):
            out_path = structure_path(temperature, index)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            seed = BASE_SEED + temperature + index

            if overwrite or not out_path.exists():
                initial = make_initial_structure(seed)
                cemc_structure = run_cemc(initial, temperature)
                cemc_structure.to(str(out_path), fmt="poscar")

            records.append({
                "composition": COMPOSITION,
                "temperature_K": temperature,
                "index": index,
                "seed": seed,
                "structure": str(out_path.relative_to(REPO_ROOT)),
            })

    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_DIR / "cemc_structure_manifest.csv", index=False)
    return df


cemc_manifest = generate_cemc_structures(overwrite=False)
cemc_manifest.head()


## Li percolation analysis

The percolation model follows the `dribble` input used in the study: Li sites are the percolating cation sublattice, Mn/Ti are treated as transition-metal cation sites, oxygen is ignored, and Li-Li connectivity is defined using the `MinCommonNNNeighborsBR` rule with two common nearest neighbors.

In [ ]:
def write_dribble_input(json_path: Path, structure_file: Path) -> None:
    content = {
        "structure": str(structure_file.resolve()),
        "formula_units": 1,
        "sublattices": {
            "cations": {
                "description": "Cation sites",
                "sites": {"species": ["Li"]},
                "initial_occupancy": {"Li": 1.0},
            },
            "cation2": {
                "description": "Transition-metal cation sites",
                "sites": {"species": ["Mn", "Ti"]},
                "initial_occupancy": {"TM": 1.0},
            },
            "oxygen": {
                "description": "Oxygen sites",
                "sites": {"species": ["O"]},
                "ignore": True,
            },
        },
        "bonds": [
            {
                "sublattices": ["cations", "cations"],
                "bond_rules": [["MinCommonNNNeighborsBR", {"num_neighbors": 2}]],
            }
        ],
        "percolating_species": ["Li"],
        "flip_sequence": [["TM", "Li"]],
    }
    json_path.write_text(json.dumps(content, indent=4), encoding="utf-8")


def percolating_li_fraction(structure_file: Path, work_dir: Path, save_clusters: bool = False) -> dict:
    work_dir.mkdir(parents=True, exist_ok=True)
    input_json = work_dir / "input-bond-rule.json"
    write_dribble_input(input_json, structure_file)

    inp = Input.from_file(str(input_json))
    lattice = Lattice.from_input_object(inp, supercell=(1, 1, 1))
    percolator = Percolator.from_input_object(inp, lattice, verbose=False)

    cwd = Path.cwd()
    os.chdir(work_dir)
    try:
        n_occupied = int(percolator.num_occupied)
        n_spanning = int(percolator.check_spanning(
            verbose=False,
            save_clusters=save_clusters,
            static_sites=inp.static_sites,
        ))
    finally:
        os.chdir(cwd)

    fraction = float(n_spanning) / float(n_occupied) if n_occupied else np.nan
    return {
        "occupied_li_sites": n_occupied,
        "percolating_li_sites": n_spanning,
        "percolating_li_fraction": fraction,
        "percolating_li_percent": 100.0 * fraction,
    }


def run_percolation(cemc_manifest: pd.DataFrame, overwrite: bool = False) -> pd.DataFrame:
    result_csv = OUTPUT_DIR / "percolation_results.csv"
    if result_csv.exists() and not overwrite:
        return pd.read_csv(result_csv)

    rows = []
    for row in cemc_manifest.to_dict("records"):
        structure_file = REPO_ROOT / row["structure"]
        work_dir = OUTPUT_DIR / f"{row['temperature_K']}K" / "percolation" / f"CEMC_{row['index']:03d}"
        metrics = percolating_li_fraction(structure_file, work_dir, save_clusters=False)
        rows.append({**row, **metrics})

    results = pd.DataFrame(rows)
    results.to_csv(result_csv, index=False)
    return results


percolation_results = run_percolation(cemc_manifest, overwrite=False)
percolation_results.head()


## Summary and Fig. 2b-style plot

The box plot below summarizes 100 structures per CEMC temperature. The y-axis is the fraction of Li sites belonging to a spanning percolation network, reported as a percentage.

In [ ]:
summary = (
    percolation_results
    .groupby("temperature_K")["percolating_li_percent"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)
summary.to_csv(OUTPUT_DIR / "percolation_summary_by_temperature.csv", index=False)
summary


In [ ]:
def plot_fig2b_style(results: pd.DataFrame) -> Path:
    ordered = list(TEMPERATURES)
    values = [
        results.loc[results["temperature_K"] == temperature, "percolating_li_percent"].dropna().to_numpy()
        for temperature in ordered
    ]

    fig, ax = plt.subplots(figsize=(5.2, 4.0), dpi=300)
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#e41a1c"]
    box = ax.boxplot(values, patch_artist=True, widths=0.55, showfliers=False)

    for patch, color in zip(box["boxes"], colors):
        patch.set(facecolor=color, alpha=0.18, edgecolor=color, linewidth=1.4)
    for key in ["whiskers", "caps", "medians"]:
        for line in box[key]:
            line.set(linewidth=1.2)

    rng = np.random.default_rng(42)
    for pos, vals, color in zip(range(1, len(values) + 1), values, colors):
        jitter = rng.normal(0, 0.045, len(vals))
        ax.scatter(np.full(len(vals), pos) + jitter, vals, s=12, facecolors="none", edgecolors=color, linewidth=0.55)
        if len(vals):
            ax.text(pos - 0.18, np.nanmean(vals) + 3, f"{np.nanmean(vals):.1f}", fontsize=9)

    ax.set_xticks(range(1, len(ordered) + 1), [f"{temperature:g} K" for temperature in ordered])
    ax.set_ylabel("Percolating Li fraction (%)")
    ax.set_xlabel("CEMC temperature")
    ax.set_ylim(0, 100)
    ax.text(0.06, 0.92, f"n = {N_STRUCTURES}", transform=ax.transAxes, fontsize=10)
    ax.tick_params(direction="out", width=1.1, length=4)
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)

    fig.tight_layout()
    out_path = OUTPUT_DIR / "Fig2b_CEMC_percolation_reproduction.png"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return out_path


figure_path = plot_fig2b_style(percolation_results)
print(f"Saved figure to: {figure_path}")


## Output files

After a full run, the notebook writes:

- `outputs/cemc_percolation_Li1p2Mn0p4Ti0p4O2/*K/structures/CEMC_*.vasp`: generated CEMC structures;
- `outputs/cemc_percolation_Li1p2Mn0p4Ti0p4O2/cemc_structure_manifest.csv`: structure manifest;
- `outputs/cemc_percolation_Li1p2Mn0p4Ti0p4O2/percolation_results.csv`: per-structure percolation results;
- `outputs/cemc_percolation_Li1p2Mn0p4Ti0p4O2/percolation_summary_by_temperature.csv`: temperature-level summary;
- `outputs/cemc_percolation_Li1p2Mn0p4Ti0p4O2/Fig2b_CEMC_percolation_reproduction.png`: Fig. 2b-style reproduction plot.

The `outputs/` directory is intentionally ignored by git because the full workflow generates hundreds of structures.